# Setup

In [ ]:
using PowerModels
using Ipopt
using JuMP

using CSV
using DataFrames
using JuMP
using Gurobi
using Ipopt
using LinearAlgebra
using Random
using Plots
using SparseArrays
using XLSX
using SCS
using Distributions
using HTTP
using Dates
using Pkg
using Gurobi
using PowerModels
using DataFrames
using Random
using Statistics  
using MathOptInterface

# Setup system data

In [ ]:
TotalDays = 731
T = 24
K = 4
all_nodes = 1:24

Random.seed!(1234)

file_loc = "/Users/mehrnoushghazanfariharandi/Library/CloudStorage/OneDrive-RutgersUniversity/RUTGERS/PHD-research/forth paper/data"
case_file = "/Users/mehrnoushghazanfariharandi/Library/CloudStorage/OneDrive-RutgersUniversity/RUTGERS/PHD-research/Secondpaper2024/case24_ieee_rts.m"

function add_day_index!(df, T)
    df[!, "Day Index"] = ceil.(Int, (1:nrow(df)) ./ T)
end

function add_day_hour!(df, T)
    n = nrow(df)
    df[!, "Day Index"] = ceil.(Int, (1:n) ./ T)
    df[!, "Hour"] = [(i - 1) % T + 1 for i in 1:n]
end


load_timeseries1 = CSV.read("$file_loc/load_timeseries1v2.csv", DataFrame)
add_day_index!(load_timeseries1, T)


data = PowerModels.parse_file(case_file)
PowerModels.standardize_cost_terms!(data, order=2)
PowerModels.calc_thermal_limits!(data)
ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

target_load_nodes = [14, 18, 7]

for bus in keys(ref[:bus_loads])
    if !(bus in target_load_nodes)
        ref[:bus_loads][bus] = []
    end
end


net_load_fixed = Dict{Int, Dict{Int, Dict{Int, Float64}}}()

for D in 1:TotalDays
    net_load_fixed[D] = Dict()
    for t in 1:T
        net_load_fixed[D][t] = Dict()
        for (i, bus) in ref[:bus]
            bus_loads = [
                ref[:load][l]["pd"] / 28.5 *
                load_timeseries1[load_timeseries1[!, "Day Index"] .== D, "real"] ./ 100
                for l in ref[:bus_loads][i] if !isempty(ref[:load][l])
            ]

            net_load_fixed[D][t][i] = isempty(bus_loads) ? 0.0 : bus_loads[1][t]
        end
    end
end


data = PowerModels.parse_file(case_file)
PowerModels.standardize_cost_terms!(data, order=2)
PowerModels.calc_thermal_limits!(data)
ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

for (l, branch) in data["branch"]
    if haskey(branch, "rate_a")
        branch["rate_a"] *= 0.9
    end
end

ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

target_load_nodes = [3, 5, 9, 16, 19, 20]

for bus in keys(ref[:bus_loads])
    if !(bus in target_load_nodes)
        ref[:bus_loads][bus] = []
    end
end


for (gen_id, gen_data) in ref[:gen]
    if gen_data["cost"][2] < 4
        original_cost = gen_data["cost"][2]
        gen_data["cost"][2] = rand(400:10:4000)
    end
end


load_timeseries = CSV.read("$file_loc/load_timeseriesv2.csv", DataFrame)
add_day_index!(load_timeseries, T)

wind_timeseries = CSV.read("$file_loc/wind_timeseries_scenario11v2.csv", DataFrame)
wind_timeseries2 = CSV.read("$file_loc/wind_timeseries_scenario21v2.csv", DataFrame)
wind_timeseries3 = CSV.read("$file_loc/wind_timeseries_scenario31v2.csv", DataFrame)
wind_timeseries4 = CSV.read("$file_loc/wind_timeseries_scenario41v2.csv", DataFrame)

wind_timeseries_real1 = CSV.read("$file_loc/wind_timeseries_real_year1v2.csv", DataFrame)
wind_timeseries_real2 = CSV.read("$file_loc/wind_timeseries_real_year2v2.csv", DataFrame)
wind_timeseries_real3 = CSV.read("$file_loc/wind_timeseries_real_year3v2.csv", DataFrame)
wind_timeseries_real4 = CSV.read("$file_loc/wind_timeseries_real_year4v2.csv", DataFrame)
wind_timeseries_real5 = CSV.read("$file_loc/wind_timeseries_real_year5v2.csv", DataFrame)

for df in [
    wind_timeseries,
    wind_timeseries2,
    wind_timeseries3,
    wind_timeseries4,
    wind_timeseries_real1,
    wind_timeseries_real2,
    wind_timeseries_real3,
    wind_timeseries_real4,
    wind_timeseries_real5
]
    add_day_hour!(df, T)
end

TotalDays = ceil(Int, nrow(wind_timeseries_real1) / T)


wind_farm_to_node_real = Dict(
    "windreal1" => 3,
    "windreal2" => 5,
    "windreal3" => 9,
    "windreal4" => 16,
    "windreal5" => 19,
    "windreal6" => 20,
    "windreal7" => 16
)

pwind_real = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for node in all_nodes
    pwind_real[node] = Dict()
    for t in 1:T
        pwind_real[node][t] = Dict()
        for D in 1:TotalDays
            pwind_real[node][t][D] = Dict()
            for k in 1:4
                pwind_real[node][t][D][k] = 0.0
            end
        end
    end
end

for (farm, node) in wind_farm_to_node_real
    for D in 1:TotalDays
        for t in 1:T
            filtered_rows = wind_timeseries_real1[(wind_timeseries_real1[!, "Day Index"] .== D) .&& (wind_timeseries_real1[!, "Hour"] .== t), :]
            power_value_real1 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real2[(wind_timeseries_real2[!, "Day Index"] .== D) .&& (wind_timeseries_real2[!, "Hour"] .== t), :]
            power_value_real2 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real3[(wind_timeseries_real3[!, "Day Index"] .== D) .&& (wind_timeseries_real3[!, "Hour"] .== t), :]
            power_value_real3 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            filtered_rows = wind_timeseries_real4[(wind_timeseries_real4[!, "Day Index"] .== D) .&& (wind_timeseries_real4[!, "Hour"] .== t), :]
            power_value_real4 = isempty(filtered_rows) ? 0.0 : (filtered_rows[1, farm] / 15000) * 400

            pwind_real[node][t][D][1] = power_value_real1
            pwind_real[node][t][D][2] = power_value_real2
            pwind_real[node][t][D][3] = power_value_real3
            pwind_real[node][t][D][4] = power_value_real4
        end
    end
end


wind_farm_to_node = Dict(
    "Wind Farm 1 (MW)" => 3,
    "Wind Farm 2 (MW)" => 5,
    "Wind Farm 3 (MW)" => 9,
    "Wind Farm 4 (MW)" => 16,
    "Wind Farm 5 (MW)" => 19,
    "Wind Farm 6 (MW)" => 20,
    "Offshore Wind (MW)" => 17
)

TotalDays = ceil(Int, nrow(wind_timeseries) / T)

pwind = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for node in all_nodes
    pwind[node] = Dict()
    for t in 1:T
        pwind[node][t] = Dict()
        for D in 1:TotalDays
            pwind[node][t][D] = Dict()
            for k in 1:4
                pwind[node][t][D][k] = 0.0
            end
        end
    end
end

for (farm, node) in wind_farm_to_node
    for D in 1:TotalDays
        for t in 1:T
            filtered_rows1 = wind_timeseries[(wind_timeseries[!, "Day Index"] .== D) .&& (wind_timeseries[!, "Hour"] .== t), :]
            power_value1 = isempty(filtered_rows1) ? 0.0 : (filtered_rows1[1, farm] / 15000) * 400

            filtered_rows2 = wind_timeseries2[(wind_timeseries2[!, "Day Index"] .== D) .&& (wind_timeseries2[!, "Hour"] .== t), :]
            power_value2 = isempty(filtered_rows2) ? 0.0 : (filtered_rows2[1, farm] / 15000) * 400

            filtered_rows3 = wind_timeseries3[(wind_timeseries3[!, "Day Index"] .== D) .&& (wind_timeseries3[!, "Hour"] .== t), :]
            power_value3 = isempty(filtered_rows3) ? 0.0 : (filtered_rows3[1, farm] / 15000) * 400

            filtered_rows4 = wind_timeseries4[(wind_timeseries4[!, "Day Index"] .== D) .&& (wind_timeseries4[!, "Hour"] .== t), :]
            power_value4 = isempty(filtered_rows4) ? 0.0 : (filtered_rows4[1, farm] / 15000) * 400

            pwind[node][t][D][1] = power_value1
            pwind[node][t][D][2] = power_value2
            pwind[node][t][D][3] = power_value3
            pwind[node][t][D][4] = power_value4
        end
    end
end


loadd = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for D in 1:TotalDays
    loadd[D] = Dict()
    for t in 1:T
        for (i, bus) in ref[:bus]
            loadd[D][t] = get(loadd[D], t, Dict())
            loadd[D][t][i] = get(loadd[D][t], i, Dict())

            for k in 1:K
                column_name = Symbol("load_forecast_k$k")

                activeload = if !isempty(ref[:bus_loads][i])
                    [
                        ref[:load][l]["pd"] / 28.5 *
                        load_timeseries[load_timeseries[!, "Day Index"] .== D, column_name]
                        for l in ref[:bus_loads][i]
                    ][1][t]
                else
                    0.0
                end

                loadd[D][t][i][k] = activeload
            end
        end
    end
end


net_load = Dict{Int, Dict{Int, Dict{Int, Dict{Int, Float64}}}}()

for D in 1:TotalDays
    net_load[D] = Dict()
    for t in 1:T
        for (i, bus) in ref[:bus]
            net_load[D][t] = get(net_load[D], t, Dict())
            net_load[D][t][i] = get(net_load[D][t], i, Dict())

            for k in 1:K
                net_load[D][t][i][k] = (loadd[D][t][i][k] - pwind[i][t][D][k]) / 100
            end
        end
    end
end


bus_ids = sort(collect(keys(ref[:bus])))
numBuses = length(bus_ids)
bus_index = Dict(bid => k for (k, bid) in enumerate(bus_ids))

maxK = 4

realload = zeros(TotalDays, T, numBuses, maxK)

for D in 1:TotalDays
    real_vec = Vector(load_timeseries1[load_timeseries1[!, "Day Index"] .== D, "real"])
    @assert length(real_vec) == T "load_timeseries1 day $D must have T rows"

    for t in 1:T
        for bid in bus_ids
            load_series = [
                (ref[:load][l]["pd"] / 28.5) .* real_vec
                for l in ref[:bus_loads][bid] if !isempty(ref[:load][l])
            ]

            base_load_t = isempty(load_series) ? 0.0 : sum(s[t] for s in load_series)

            nK = haskey(pwind_real, bid) &&
                 haskey(pwind_real[bid], t) &&
                 haskey(pwind_real[bid][t], D) ? length(pwind_real[bid][t][D]) : 0

            for k in 1:maxK
                wind_t = k <= nK ? pwind_real[bid][t][D][k] : 0.0
                realload[D, t, bus_index[bid], k] = (base_load_t - wind_t) / 100.0
            end
        end
    end
end

# Capacity data

In [ ]:
scale1 = 0.0001


cand_wind_data = Dict(
    "Wind Gen 1" => (bus=3,  pmax=18, c_fix=5000000 * scale1, c_var=9000000 * scale1),
     "Wind Gen 2" => (bus=5, pmax=15, c_fix=4000000  * scale1, c_var=8000000 * scale1),
    "Wind Gen 3" => (bus=9, pmax=20, c_fix=3500000 * scale1, c_var=8500000 * scale1),
    "Wind Gen 4" => (bus=16,  pmax=15, c_fix=5000000 * scale1, c_var=9000000 * scale1),
    "Wind Gen 5" => (bus=19, pmax=20, c_fix=4000000 * scale1, c_var=10000000 * scale1),
     "Wind Gen 6" => (bus=20, pmax=14, c_fix=3500000 * scale1, c_var=8500000 * scale1)
)
RENW = collect(keys(cand_wind_data))
bus_of_wind = Dict(b => cand_wind_data[b].bus for b in RENW)



battery_data = Dict(
    "Battery 16" => (bus = 16, E = 6.4, p_ch = 1.600, p_dis = 1.60, c_var_b=105000 * scale1, c_var_E = 12000000 * scale1, c_opr_b = 150 * scale1),
    "Battery 20" => (bus = 20, E = 3.8, p_ch = 0.950, p_dis = 0.95, c_var_b=110000 * scale1,c_var_E = 10000000 * scale1, c_opr_b=250 * scale1 ),
    "Battery 7"  => (bus = 7,  E =  1.6, p_ch = 0.4000, p_dis = 0.40, c_var_b=100000 * scale1 ,c_var_E = 11000000 * scale1, c_opr_b = 400* scale1),
    "Battery 24" => (bus = 24, E = 2, p_ch = 0.50, p_dis = 0.50, c_var_b=80000 * scale1, c_var_E = 17000000 * scale1, c_opr_b= 500 * scale1),
)
BATS = collect(keys(battery_data))
bus_of_bat = Dict(b => battery_data[b].bus for b in BATS)




fuel_gen_data = Dict(
    "Fuel Gen 1" => (bus=8,  pmax=3.500,  c_fix=38000.0 * scale1, c_var=4000000* scale1, c_can_fuel=2000.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 2" => (bus=21,  pmax=2.800,  c_fix=39000.0 * scale1, c_var=4200000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 3" => (bus=9,  pmax=3.1000, c_fix=14000.0 * scale1, c_var=2500000 * scale1, c_can_fuel=2000.0 * scale1, ru=0.8, rd=0.8),
   "Fuel Gen 4" => (bus=17, pmax=4.1200, c_fix=20000.0 * scale1, c_var=2500000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 5" => (bus=19, pmax=3.1500, c_fix=15500.0 * scale1, c_var=2500000 * scale1, c_can_fuel=1700.0 * scale1, ru=0.8 , rd=0.8 ),
    "Fuel Gen 6" => (bus=20, pmax=2.2000, c_fix=18000.0 * scale1, c_var=3400000 * scale1, c_can_fuel=2100.0 * scale1, ru=0.8 , rd=0.8),
    "Fuel Gen 7" => (bus=16, pmax=4.750,  c_fix=36500.0 * scale1, c_var=3500000 * scale1, c_can_fuel=1200.0 * scale1, ru=0.8 , rd=0.8)
)

_get(g, sym::Symbol, default=nothing) =
    hasproperty(g, sym) ? getfield(g, sym) :
    g isa AbstractDict   ? get(g, String(sym), default) :
    default


_get(g, sym::Symbol, default=nothing) =
    hasproperty(g, sym) ? getfield(g, sym) :
    g isa AbstractDict   ? get(g, String(sym), default) :
    default

function merge_candidates_into_ref!(ref::Dict{Symbol,Any},
                                    fuel_gen_data::AbstractDict{<:AbstractString,<:Any})
    name_to_id = Dict{String,Int}()
    existing_ids = collect(keys(ref[:gen]))
    next_id = isempty(existing_ids) ? 1 : maximum(existing_ids) + 1

    for name in keys(fuel_gen_data)
        g = fuel_gen_data[name]
        bus  = _get(g, :bus)
        pmax = _get(g, :pmax)
        cvar = _get(g, :c_var)
        cfuel = _get(g, :c_can_fuel)
        isnothing(bus)  && error("Generator '$name' missing field :bus")
        isnothing(pmax) && error("Generator '$name' missing field :pmax")
        isnothing(cvar) && error("Generator '$name' missing field :c_var")
        isnothing(cfuel) && error("Generator '$name' missing field :c_can_fuel")
        pmin = something(_get(g, :pmin, nothing), 0.0)

        gid = next_id; next_id += 1
        ref[:gen][gid] = Dict(
            "gen_bus"   => bus,
            "pmin"      => pmin,
            "pmax"      => pmax,
            "status"    => 1,
            "name"      => name,
            "cost"      => [0.0, cfuel],
            "candidate" => 1
        )
        push!(get!(ref[:bus_gens], bus, Int[]), gid)
        name_to_id[name] = gid
    end
    return name_to_id
end

gen_id_map = merge_candidates_into_ref!(ref, fuel_gen_data)
NEWGENS = collect(keys(fuel_gen_data))

# cost maps
c_g_fix_id = Dict(id => fuel_gen_data[name].c_fix for (name, id) in gen_id_map)
c_g_var_id = Dict(id => fuel_gen_data[name].c_var for (name, id) in gen_id_map)
CAND = sort!(collect(keys(c_g_fix_id)))
@assert CAND == sort(collect(keys(c_g_var_id)))

# sets
ALLGENS = sort!(collect(keys(ref[:gen])))
EXIST   = setdiff(ALLGENS, CAND)

# candidate lines
CANDLINES = Dict(
    "CL_1" => (f_bus = 3,  t_bus = 9,  x_pu = 0.12, rate_MW = 2.000 * 0.9, c_fix = 20000.0 * scale1, c_var = 10000 * scale1),
    "CL_2" => (f_bus = 16, t_bus = 19, x_pu = 0.10, rate_MW = 2.500 * 0.9, c_fix = 22000.0 * scale1, c_var = 9000.5 * scale1)
)
CL   = collect(keys(CANDLINES))
fbus = Dict(l => CANDLINES[l].f_bus for l in CL)
tbus = Dict(l => CANDLINES[l].t_bus for l in CL)
Bx   = Dict(l => 1.0 / CANDLINES[l].x_pu for l in CL)
Fmax = Dict(l => CANDLINES[l].rate_MW for l in CL)
c_l_fix = Dict(l => CANDLINES[l].c_fix for l in CL)
c_l_var = Dict(l => CANDLINES[l].c_var for l in CL)

function merge_candidate_lines_into_ref!(ref, CANDLINES)
    name_to_id = Dict{String,Int}()
    existing = collect(keys(ref[:branch]))
    next_id = isempty(existing) ? 1 : maximum(existing) + 1
    for name in keys(CANDLINES)
        cl = CANDLINES[name]
        f  = cl.f_bus; t = cl.t_bus; x = cl.x_pu; F = cl.rate_MW
        l_id = next_id; next_id += 1
        ref[:branch][l_id] = Dict(
            "name"      => name,
            "f_bus"     => f,
            "t_bus"     => t,
            "br_r"      => 0.0,
            "br_x"      => x,
            "br_b"      => 0.0,
            "tap"       => 1.0,
            "shift"     => 0.0,
            "rate_a"    => F,
            "status"    => 1,
            "candidate" => 1,
            "c_fix"     => cl.c_fix,
            "c_var"     => cl.c_var
        )
        push!(ref[:arcs_from], (l_id, f, t))
        push!(get!(ref[:bus_arcs], f, Vector{Tuple{Int,Int,Int}}()), (l_id, f, t))
        push!(get!(ref[:bus_arcs], t, Vector{Tuple{Int,Int,Int}}()), (l_id, t, f))
        name_to_id[name] = l_id
    end
    return name_to_id
end

line_id_map = merge_candidate_lines_into_ref!(ref, CANDLINES)
c_l_fix_id = Dict(id => CANDLINES[name].c_fix for (name, id) in line_id_map)
c_l_var_id = Dict(id => CANDLINES[name].c_var for (name, id) in line_id_map)
CANDL = sort!(collect(keys(c_l_fix_id)))
@assert CANDL == sort(collect(keys(c_l_var_id)))

ref[:arcs_from] = Tuple{Int,Int,Int}[]
ref[:bus_arcs]  = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
for (l, br) in ref[:branch]
    f = br["f_bus"]; to = br["t_bus"]
    push!(ref[:arcs_from], (l, f, to))
    push!(get!(ref[:bus_arcs], f, Vector{Tuple{Int,Int,Int}}()), (l, f, to))
    push!(get!(ref[:bus_arcs], to, Vector{Tuple{Int,Int,Int}}()), (l, to, f))
end

ALLLINES = sort!(collect(keys(ref[:branch])))
EXISTL   = setdiff(ALLLINES, CANDL)

B_line = Dict{Int,Float64}()
for (l, br) in ref[:branch]
    g, b = PowerModels.calc_branch_y(br)
    B_line[l] = b
end


Gens       = collect(ALLGENS)
CandGens   = collect(CAND)
ExistGens  = collect(EXIST)
Lines      = collect(ALLLINES)
CandLines  = collect(CANDL)
ExistLines = collect(EXISTL)


# Uncertainty-agnostic benchmark planning model

In [ ]:
const MOI = MathOptInterface

# ======================= MODEL  =======================

M = 100000
c_cur_prime=2000 * scale1
c_shed_prime = 1000000 * scale1
c_cur=2000 * scale1
c_shed = 1000000 * scale1
TotalDays = 365
r = 0.25
TotalYears = 1
T=24
DD = 1:2:TotalDays

model = Model(Gurobi.Optimizer)


eta = [0.6, 0.20, 0.20, 0.20]
loadyearly = [1, 1.15, 2.1, 3.15]

@variable(model, eps >= 0)
fix(eps, 1.0e4; force = true) 

# ======================= variables =======================
@variable(model, va_prime_blnc[i in keys(ref[:bus]), t in 1:T, D in DD, y in 1:TotalYears])
@variable(model, p_new[g in CandGens, y in 1:TotalYears] >= 0)
@variable(model, f_cand[l in CandLines,y in 1:TotalYears])

@variable(model, p_wind_cap[g in RENW, y in 1:TotalYears] >= 0)
@variable(model, p_shed_prime_blnc[i in keys(ref[:bus]), t in 1:T, D in DD, y in 1:TotalYears]>= 0)
@variable(model, p_cur_prime_blnc[i in keys(ref[:bus]), t in 1:T, D in DD, y in 1:TotalYears]>=0)

@variable(model, pg[i in Gens, t in 1:T, D in DD, y in 1:TotalYears])
@variable(model, windpower_rt[w in RENW, t=1:T, D in DD, y in 1:TotalYears]>= 0)



DF = Dict(y => 1 / (1 + r)^(y) for y in 1:TotalYears)


gen_var_cost = Dict(
    i => (i in CandGens ?
        get(ref[:gen][i], "cost", [0.0, 0.0])[2]  :
        get(ref[:gen][i], "cost", [0.0, 0.0])[2] * scale1 )
    for i in Gens
)


c_fuel_id = Dict{Int,Float64}()
for (name, gid) in gen_id_map
    cval = get(fuel_gen_data[name], :c_fuel, get(fuel_gen_data[name], :c_can_fuel, nothing))
    isnothing(cval) && error("Candidate '$name' missing :c_fuel (or :c_can_fuel) value.")
    c_fuel_id[gid] = cval
end

r_cost_up = Dict(i => 1.5 * gen_var_cost[i] for i in Gens)
r_cost_dn = Dict(i => 0.7 * gen_var_cost[i] for i in Gens)


@constraint(model, pnew_cap[g in CandGens, y in 1:TotalYears], p_new[g,y] <= ref[:gen][g]["pmax"])
if TotalYears >= 2
    @constraint(model,
        pnew_monotone[g in CandGens, y in 2:TotalYears],
        p_new[g,y-1] <= p_new[g,y]
    )
end
@variable(model, -ref[:branch][l]["rate_a"] <=
    p_prime_blnc[(l,i,j) in ref[:arcs_from], t in 1:T, d in DD, y in 1:TotalYears] <=
    ref[:branch][l]["rate_a"])

p_expr_prime_blnc = Dict(((l,i,j), t,d,y) => 1.0 * p_prime_blnc[(l,i,j),t,d,y]
    for (l,i,j) in ref[:arcs_from], t in 1:T, d in DD, y in 1:TotalYears)
p_expr_prime_blnc = merge(p_expr_prime_blnc,
    Dict(((l,j,i), t,d,y) => -1.0 * p_prime_blnc[(l,i,j),t,d,y]
        for (l,i,j) in ref[:arcs_from], t in 1:T, d in DD, y in 1:TotalYears))

for d in DD, t in 1:T, y in 1:TotalYears, l in ExistLines
    br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
    con = @constraint(model, p_prime_blnc[f_idx,t,d,y] == -B_line[l]*(va_prime_blnc[f,t,d,y] - va_prime_blnc[to,t,d,y]))
    set_name(con, "dc_blnc_exist[l=$l,t=$t,d=$d,y=$y]")
end
for d in DD, t in 1:T, y in 1:TotalYears, l in CandLines
    br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
    con = @constraint(model, p_prime_blnc[f_idx,t,d,y] == -B_line[l]*(va_prime_blnc[f,t,d,y] - va_prime_blnc[to,t,d,y]))
    set_name(con, "dc_blnc_cand[l=$l,t=$t,d=$d,y=$y]")
end

for l in CandLines, y in 2:TotalYears
    @constraint(model, f_cand[l,y-1] <= f_cand[l,y])
end
for l in CandLines, y in 1:TotalYears
    F = ref[:branch][l]["rate_a"]
    @constraint(model, f_cand[l,y] <= F)
    @constraint(model, 0 <= f_cand[l,y])
end


for l in CandLines, d in DD, t in 1:T, y in 1:TotalYears
    br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
    @constraint(model, p_prime_blnc[f_idx,t,d,y] <=  f_cand[l,y])
    @constraint(model, -f_cand[l,y] <=  p_prime_blnc[f_idx,t,d,y])
end



for g in RENW, y in 1:TotalYears
    @constraint(model, p_wind_cap[g,y] <= cand_wind_data[g].pmax)
    @constraint(model, 0 <= p_wind_cap[g,y])
end
for g in RENW, y in 2:TotalYears
    @constraint(model, p_wind_cap[g,y-1] <= p_wind_cap[g,y])
end

for g in RENW,  y in 1:TotalYears, d in DD, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model,
        windpower_rt[g,t,d,y] <= (pwind_real[bus][t][d][y] / 400)* p_wind_cap[g,y]
    )
    @constraint(model, 0 <= windpower_rt[g,t,d,y])
end


BATS = collect(keys(battery_data))

@variable(model, e_rt[g in BATS, t in 1:T, D in DD, y in 1:TotalYears])
@variable(model, P_BAT_rt[g in BATS, t in 1:T, D in DD, y in 1:TotalYears] )
@variable(model, P_BAT_cap[g in BATS, y in 1:TotalYears] )
@variable(model, E_BAT_cap[g in BATS, y in 1:TotalYears] )

for g in BATS, y in 2:TotalYears
    @constraint(model, E_BAT_cap[g,y-1] <= E_BAT_cap[g,y])
    @constraint(model, P_BAT_cap[g,y-1] <= P_BAT_cap[g,y] )
end
for g in BATS, y in 1:TotalYears
    @constraint(model, E_BAT_cap[g,y] <= battery_data[g].E)
    @constraint(model, 0 <= E_BAT_cap[g,y])
end

for g in BATS, y in 1:TotalYears
    @constraint(model, E_BAT_cap[g,y] == 4 * P_BAT_cap[g,y] )
end

for g in BATS, d in DD, t in 1:T, y in 1:TotalYears
    @constraint(model, e_rt[g,t,d,y] <= E_BAT_cap[g,y])
    @constraint(model, 0 <= e_rt[g,t,d,y])
end


@variable(model, P_DIS_rt[g in BATS, t in 1:T, d in DD, y in 1:TotalYears] >= 0)
@variable(model, P_CH_rt[g in BATS,t in 1:T, d in DD, y in 1:TotalYears] >= 0)


for g in BATS, t in 1:T, d in DD, y in 1:TotalYears
    @constraint(model, P_BAT_rt[g,t,d,y] <= P_BAT_cap[g,y])
    @constraint(model, -P_BAT_cap[g,y] <= P_BAT_rt[g,t,d,y])
    @constraint(model, P_BAT_rt[g,t,d,y] == P_DIS_rt[g,t,d,y] - P_CH_rt[g,t,d,y])
end


for g in BATS, y in 1:TotalYears
    @constraint(model, P_BAT_cap[g,y] <= battery_data[g].p_dis)
    @constraint(model, 0<= P_BAT_cap[g,y])
end
for g in BATS, d in DD, t in 2:T, y in 1:TotalYears
    @constraint(model, e_rt[g,t,d,y] == e_rt[g,t-1,d,y] - P_BAT_rt[g,t,d,y])
end
@constraint(model, [b in BATS, d in DD, y in 1:TotalYears], e_rt[b,1,d,y] == 0.5 * E_BAT_cap[b,y])
@constraint(model, [b in BATS, d in DD, y in 1:TotalYears], e_rt[b,T,d,y] == 0.5 * E_BAT_cap[b,y])

WIND_NAMES   = Set(keys(wind_farm_to_node))
RENEWABLE_IDS = [gid for gid in Gens if get(ref[:gen][gid], "name", nothing) in WIND_NAMES]



@variable(model, ren_shortfall[y in 1:TotalYears] >= 0)

for y in 1:TotalYears
    @constraint(model,
        eta[y] * (sum(sum(
            sum(pg[g,t,D,y] for g in Gens)
            for t in 1:T
        ) + sum(
            sum(windpower_rt[w,t,D,y] for w in RENW)
            for t in 1:T
        ) for D in DD))
        <=
        sum(sum(
            sum(windpower_rt[w,t,D,y] for w in RENW)
            for t in 1:T
        ) for D in DD)+ ren_shortfall[y]
    )
end


avg_wind_inv_price = mean(cand_wind_data[g].c_fix for g in RENW)
penalty_wind = 5 * avg_wind_inv_price


# --- Nodal balance ---
for y in 1:TotalYears, d in DD, t in 1:T
    for (i, _bus) in ref[:bus]
        con = @constraint(model,
            sum(p_expr_prime_blnc[a, t, d, y] for a in ref[:bus_arcs][i]) ==
            sum(pg[g,t,d,y]  for g in ref[:bus_gens][i])
            +sum(P_BAT_rt[b, t, d, y] for b in BATS if bus_of_bat[b] == i)
            - (loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i])
            + sum( windpower_rt[g,t,d,y] for g in RENW if bus_of_wind[g] == i)
            + p_shed_prime_blnc[i,t,d,y] 
        )
        set_name(con, "nb_blnc[i=$i,t=$t,d=$d,y=$y]")
    end
end

for g in CandGens, d in DD, t in 1:T, y in 1:TotalYears
    @constraint(model, pg[g,t,d,y]  <= p_new[g,y])
    @constraint(model, 0 <= pg[g,t,d,y] )
   
end
for g in ExistGens, d in DD, t in 1:T, y in 1:TotalYears
    @constraint(model, pg[g,t,d,y]  <= ref[:gen][g]["pmax"])
    @constraint(model, 0 <= pg[g,t,d,y] )
end


#ramp
for g in CandGens, d in DD, y in 1:TotalYears, t=2:T
    @constraint(model, pg[g,t,d,y] - pg[g,t-1,d,y] <= 0.8 * p_new[g,y])
    @constraint(model, pg[g,t-1,d,y] - pg[g,t,d,y] <= p_new[g,y])
end


for g in ExistGens, d in DD, y in 1:TotalYears, t=2:T
        @constraint(model, pg[g,t,d,y] - pg[g,t-1,d,y] <= ref[:gen][g]["pmax"]*0.8 )
        @constraint(model, pg[g,t,d,y] - pg[g,t+1,d,y] <= ref[:gen][g]["pmax"])
end


for y in 1:TotalYears, d in DD, t in 1:T, i in keys(ref[:bus])
        @constraint(model, p_shed_prime_blnc[i,t,d,y] <= loadyearly[y] * (loadd[d][t][i][y]/100) + loadyearly[y] * net_load_fixed[d][t][i])
    end
    

for y in 1:TotalYears, d in DD, t in 1:T, i in keys(ref[:bus])
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model, p_cur_prime_blnc[i,t,d,y] == 0)
    else
        @constraint(model,
            p_cur_prime_blnc[i,t,d,y] ==
            sum((pwind_real[i][t][d][y] / 400) * p_wind_cap[g,y] - windpower_rt[g,t,d,y]
                for g in gens_at_bus)
        )
    end
end





branch_b = Dict()
for (i, branch) in ref[:branch]
    g, b = PowerModels.calc_branch_y(branch)
    branch_b[i] = b
end
bus_ids = collect(keys(ref[:bus])); bus_map = Dict(bus_id => idx for (idx, bus_id) in enumerate(bus_ids))
line_ids = collect(Lines); line_map = Dict(l => k for (k,l) in enumerate(line_ids))
A = zeros(Float64, length(line_ids), length(bus_ids))
for (k,l) in enumerate(line_ids)
    br = ref[:branch][l]; f_bus = br["f_bus"]; t_bus = br["t_bus"]
    A[k, bus_map[f_bus]] =  1.0
    A[k, bus_map[t_bus]] = -1.0
end




# angle reference (lower)
for (i,_bus) in ref[:ref_buses], d in DD, t in 1:T, y in 1:TotalYears
    @constraint(model, va_prime_blnc[i,t,d,y] == 0)
end


@expression(model, investment_cost[y in 1:TotalYears],
      sum(p_new[i,y] * c_g_var_id[i] for i in CandGens)
    + sum(f_cand[l,y] * c_l_var_id[l] for l in CandLines)
    + sum(p_wind_cap[g,y] * cand_wind_data[g].c_var for g in RENW)
    + sum(battery_data[g].c_var_E * E_BAT_cap[g,y] for g in BATS)
)

@expression(model, operation_cost[y in 1:TotalYears],
      ((1/length(DD)) * sum(
            sum(
                sum(gen_var_cost[i] * pg[i,t,D,y]  for i in ALLGENS)
              + sum(
                    c_shed_prime * p_shed_prime_blnc[b,t,D,y] +
                    c_cur_prime  * p_cur_prime_blnc[b,t,D,y]
                    for b in keys(ref[:bus])
                )
              + sum(battery_data[b].c_opr_b * (P_DIS_rt[b,t,D,y] +  P_CH_rt[b,t,D,y]) for b in BATS)
                for t in 1:T
            )
            for D in DD
      )) * 365
    + (penalty_wind ) * ren_shortfall[y] 
)

@objective(model, Min,
    sum(
        DF[y] * (investment_cost[y] + operation_cost[y])
        for y in 1:TotalYears
    )
)

set_optimizer_attribute(model, "DualReductions", 0)
optimize!(model)

term_pg = value(sum( pg[i,t,D,y]
    for i in ALLGENS, t in 1:T, D in DD, y in 1:TotalYears))

term_shed = value(sum( p_shed_prime_blnc[b,t,D,y]
    for b in keys(ref[:bus]), t in 1:T, D in DD, y in 1:TotalYears))

term_cur = value(sum( p_cur_prime_blnc[b,t,D,y]
    for b in keys(ref[:bus]), t in 1:T, D in DD, y in 1:TotalYears))

term_bat = value(sum((P_DIS_rt[b,t,D,y] + P_CH_rt[b,t,D,y])
    for b in BATS, t in 1:T, D in DD, y in 1:TotalYears))



println("------ OPERATION COST BREAKDOWN ------")
println("Generation cost = ", term_pg)
println("Load shedding cost = ", term_shed)
println("Curtailment cost = ", term_cur)
println("Battery cost = ", term_bat)


println("\n========= GENERATOR COST TABLE =========")

for g in keys(r_cost_up)

    println(
        "Gen ", g,
        " | energy = ", gen_var_cost[g],
        " | r_up = ", r_cost_up[g],
        " | r_down = ", r_cost_dn[g]
    )
end
println("========================================")



In [ ]:
model_inv = model  
optimize!(model_inv)

term_inv = termination_status(model_inv)
println("Investment model status: ", term_inv)
println("Investment objective:   ", objective_value(model_inv))


y = 1  

p_new_star = Dict((g => value(p_new[g, y])) for g in CandGens)
f_cand_star = Dict((l => value(f_cand[l, y])) for l in CandLines)
p_wind_cap_star = Dict((g => value(p_wind_cap[g, y])) for g in RENW)

P_BAT_cap_star = Dict((b => value(P_BAT_cap[b, y])) for b in BATS)
E_BAT_cap_star = Dict((b => value(E_BAT_cap[b, y])) for b in BATS)

println("p_new*: ", p_new_star)
println("f_cand*: ", f_cand_star)
println("p_wind_cap*: ", p_wind_cap_star)
println("P_BAT_cap*: ", P_BAT_cap_star)
println("E_BAT_cap*: ", E_BAT_cap_star)


inv_val_by_year = value.(investment_cost)
opr_val_by_year = value.(operation_cost)

total_inv = sum(value(DF[y]) * value(investment_cost[y]) for y in 1:TotalYears)
total_opr = sum(value(DF[y]) * value(operation_cost[y]) for y in 1:TotalYears)

println("Investment cost by year = ", inv_val_by_year)
println("Operation cost by year  = ", opr_val_by_year)
println("Total discounted investment cost = ", total_inv)
println("Total discounted operation cost  = ", total_opr)
println("Total objective value            = ", objective_value(model_inv))

# Two stage testing case

In [ ]:

"""
    solve_DA_for_day(d0;
        T::Int = 24,
        y::Int = 1,
        p_new_star::Dict{Int,Float64},
        f_cand_star::Dict{Int,Float64},
        p_wind_cap_star::Dict{String,Float64},
        P_BAT_cap_star::Dict{String,Float64},
        E_BAT_cap_star::Dict{String,Float64},
        loadyearly_vec::Vector{Float64} = loadyearly)

Builds and solves the Day-Ahead (DA) problem for a single day `d0`.

Inputs:
  - d0                  : day index 
  - T                   : number of hours (default 24)
  - y                   : year index 
  - p_new_star[g]       : candidate gen capacities from investment model
  - f_cand_star[l]      : candidate line capacities from investment model
  - p_wind_cap_star[g]  : candidate wind capacities from investment model
  - P_BAT_cap_star[b]   : battery power capacities from investment model
  - E_BAT_cap_star[b]   : battery energy capacities from investment model
  - loadyearly_vec      : the loadyearly array (e.g. [1.4, 1.15, ...])

Uses:
  Global data structures:
    ref, Gens, CandGens, ExistGens, Lines, CandLines, ExistLines,
    B_line, BATS, RENW, bus_of_wind, bus_of_bat,
    net_load, net_load_fixed, loadd, pwind, pwind_real,
    battery_data, cand_wind_data,
    c_shed_prime, c_cur_prime, eta

Returns:
  - obj_DA::Float64
  - pg_DA::Dict{Tuple{Int,Int},Float64} where key = (g, t)
  - model_DA::Model
"""
function solve_DA_for_day(d0::Int;
    T::Int = 24,
    y::Int = 1,
    p_new_star::Dict{Int,Float64},
    f_cand_star::Dict{Int,Float64},
    p_wind_cap_star::Dict{String,Float64},
    P_BAT_cap_star::Dict{String,Float64},
    E_BAT_cap_star::Dict{String,Float64},
    loadyearly_vec::Vector{Float64} = loadyearly
)

    ly = loadyearly_vec[y]

    # ------------------ Build model ------------------
    model_DA = Model(Gurobi.Optimizer)

    # ====== Variables ======
    @variable(model_DA, va_prime[i in keys(ref[:bus]), t in 1:T])

    # Generation schedule
    @variable(model_DA, pg[g in Gens, t in 1:T])

    @variable(model_DA, p_new_DA[g in CandGens] >= 0)
    @variable(model_DA, f_cand_DA[l in CandLines])
    @variable(model_DA, p_wind_cap_DA[w in RENW] >= 0)
    @variable(model_DA, P_BAT_cap_DA[b in BATS] >= 0)
    @variable(model_DA, E_BAT_cap_DA[b in BATS] >= 0)

    # Fix investment variables to the planning solution
    for g in CandGens
        fix(p_new_DA[g], p_new_star[g]; force = true)
    end
    for l in CandLines
        fix(f_cand_DA[l], f_cand_star[l]; force = true)
    end
    for w in RENW
        fix(p_wind_cap_DA[w], p_wind_cap_star[w]; force = true)
    end
    for b in BATS
        fix(P_BAT_cap_DA[b], P_BAT_cap_star[b]; force = true)
        fix(E_BAT_cap_DA[b], E_BAT_cap_star[b]; force = true)
    end

    # Battery DA operation
    @variable(model_DA, e_da[b in BATS, t in 1:T])
    @variable(model_DA, P_BAT_da[b in BATS, t in 1:T])

    # Load shedding and wind curtailment
    @variable(model_DA, p_shed_prime[i in keys(ref[:bus]), t in 1:T] >= 0)
    @variable(model_DA, p_cur_prime[i in keys(ref[:bus]), t in 1:T]  >= 0)
    #wind
    @variable(model_DA, windp_da[i in RENW, t in 1:T]  >= 0)


    
    # Line flows
    @variable(model_DA,
        -ref[:branch][l]["rate_a"] <=
        p_prime[(l,i,j) in ref[:arcs_from], t in 1:T] <=
        ref[:branch][l]["rate_a"]
    )

    p_expr_prime = Dict(((l, i, j), t) => 1.0 * p_prime[(l, i, j), t]
                        for (l, i, j) in ref[:arcs_from], t in 1:T)
    p_expr_prime = merge(p_expr_prime,
        Dict(((l, j, i), t) => -1.0 * p_prime[(l, i, j), t]
            for (l, i, j) in ref[:arcs_from], t in 1:T))

   # ====== Constraints ======

    # --- DC power flow ---
    for l in ExistLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_DA,
            p_prime[f_idx, t] == -B_line[l] * (va_prime[f_bus, t] - va_prime[t_bus, t])
        )
    end
    for l in CandLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_DA,
            p_prime[f_idx, t] == -B_line[l] * (va_prime[f_bus, t] - va_prime[t_bus, t])
        )
    end

    # --- Angle reference bus ---
    for (i, _bus) in ref[:ref_buses], t in 1:T
        @constraint(model_DA, va_prime[i, t] == 0)
    end

    # --- Generation limits ---
    for g in CandGens, t in 1:T
        @constraint(model_DA, pg[g, t] <= p_new_star[g])
        @constraint(model_DA, 0 <= pg[g, t])
    end
    for g in ExistGens, t in 1:T
        @constraint(model_DA, pg[g, t] <= ref[:gen][g]["pmax"])
        @constraint(model_DA, 0 <= pg[g, t])
    end

   #ramp

    for g in CandGens, t in 2:T
        @constraint(model_DA, pg[g, t] - pg[g, t-1]<= 0.8 * p_new_star[g])
        @constraint(model_DA, pg[g,t-1] - pg[g,t] <= 0.8 * p_new_star[g])
    end


    for g in ExistGens, t in 2:T
          @constraint(model_DA, pg[g, t] - pg[g, t-1]<= 0.8*  ref[:gen][g]["pmax"] )
          @constraint(model_DA, pg[g,t-1] - pg[g,t] <= 0.8 *  ref[:gen][g]["pmax"] )
    end
   

    # candidate line gating (lower)
    for l in CandLines, t in 1:T
        br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
        @constraint(model_DA, p_prime[f_idx,t] <=  f_cand_DA[l])
        @constraint(model_DA, -f_cand_DA[l] <=  p_prime[f_idx,t])
    end
    # --- Battery DA constraints ---
    @variable(model_DA, P_DIS_da[b in BATS, t in 1:T] >= 0)
    @variable(model_DA, P_CH_da[b in BATS, t in 1:T] >= 0)
    for b in BATS, t in 1:T
        @constraint(model_DA, P_BAT_da[b,t] == P_DIS_da[b,t] - P_CH_da[b,t])
    end

    for b in BATS, t in 1:T
        @constraint(model_DA, e_da[b, t] <= E_BAT_cap_star[b])
        @constraint(model_DA, 0 <= e_da[b, t])

        @constraint(model_DA, P_BAT_da[b, t] <= P_BAT_cap_star[b])
        @constraint(model_DA, -P_BAT_cap_star[b] <= P_BAT_da[b, t])
    end

    for b in BATS, t in 2:T
        @constraint(model_DA, e_da[b, t] == e_da[b, t-1] - P_BAT_da[b, t])
    end

    @constraint(model_DA, [b in BATS],
        e_da[b, 1] == 0.5 * E_BAT_cap_star[b]
    )
    @constraint(model_DA, [b in BATS],
        e_da[b, T] == 0.5 * E_BAT_cap_star[b]
    )


    
for g in RENW, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model_DA,
        windp_da[g,t] <= (pwind[bus][t][d0][y] *1 / 400)* p_wind_cap_star[g]
    )
    @constraint(model_DA, 0 <= windp_da[g,t])
end
    
   
    # --- Nodal balance for this day d0 ---
    for (i, _bus) in ref[:bus], t in 1:T
        @constraint(model_DA,
            sum(p_expr_prime[a, t] for a in ref[:bus_arcs][i]) ==
            sum(pg[g, t] for g in ref[:bus_gens][i])
            + sum(P_BAT_da[b, t] for b in BATS if bus_of_bat[b] == i)
            -   (ly * (loadd[d0][t][i][y]/100) + ly * net_load_fixed[d0][t][i])
            + sum(windp_da[w,t] for w in RENW if bus_of_wind[w] == i)
            + p_shed_prime[i, t]
        )
    end


    for (i,_) in ref[:bus], t in 1:T
        @constraint(model_DA, p_shed_prime[i,t] <= ly * (loadd[d0][t][i][y]/100) + loadyearly[y] * net_load_fixed[d0][t][i])
    end
for t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model_DA, p_cur_prime[i,t] == 0)
    else
        @constraint(model_DA,
            p_cur_prime[i,t] ==
            sum((pwind[i][t][d0][y]*1 / 400) * p_wind_cap_star[g] - windp_da[g,t]
                for g in gens_at_bus)
        )
    end
end

 # ====== Objective: DA expected cost for this day ======

    @objective(model_DA, Min,
        sum(gen_var_cost[g] * pg[g, t] for g in Gens, t in 1:T)
        +
        sum(c_shed_prime * p_shed_prime[i, t] +
            c_cur_prime  * p_cur_prime[i, t]
            for i in keys(ref[:bus]), t in 1:T)
        +
        sum(battery_data[b].c_opr_b * (P_DIS_da[b, t] + P_CH_da[b,t])
            for b in BATS, t in 1:T)
    )

    # ------------------ Solve ------------------
    optimize!(model_DA)


da_gen_cost = value(sum(gen_var_cost[g] * pg[g,t] for g in Gens, t in 1:T))
da_shed_cost = value(sum(c_shed_prime * p_shed_prime[i,t] for i in keys(ref[:bus]), t in 1:T))
da_cur_cost = value(sum(c_cur_prime * p_cur_prime[i,t] for i in keys(ref[:bus]), t in 1:T))
da_bat_cost = value(sum(battery_data[b].c_opr_b * (P_DIS_da[b,t] + P_CH_da[b,t]) for b in BATS, t in 1:T))

println("DA day $d0 gen   = ", da_gen_cost)
println("DA day $d0 shed  = ", da_shed_cost)
println("DA day $d0 cur   = ", da_cur_cost)
println("DA day $d0 bat   = ", da_bat_cost)
println("DA day $d0 sum   = ", da_gen_cost + da_shed_cost + da_cur_cost + da_bat_cost )

    obj_DA = objective_value(model_DA)

    # Extract pg schedule for use in RT
    pg_DA = Dict{Tuple{Int,Int},Float64}()
    for g in Gens, t in 1:T
        pg_DA[(g, t)] = value(pg[g, t])
    end

    return obj_DA, pg_DA, model_DA
end


##############################################################################
# REAL-TIME (RT) MODEL FOR A SINGLE DAY 


"""
    solve_RT_for_day(d0;
        T::Int = 24,
        y::Int = 1,
        p_new_star::Dict{Int,Float64},
        p_wind_cap_star::Dict{String,Float64},
        P_BAT_cap_star::Dict{String,Float64},
        E_BAT_cap_star::Dict{String,Float64},
        r_cost_up::Dict{Int,Float64},
        r_cost_dn::Dict{Int,Float64},
        pg_DA::Dict{Tuple{Int,Int},Float64},
        loadyearly_vec::Vector{Float64} = loadyearly)

Builds and solves the Real-Time (RT) problem for a single day `d0`.

Inputs:
  - d0                    : day index 
  - T                     : number of hours (default 24)
  - y                     : year index (default 1)
  - p_new_star[g]         : candidate gen capacities from investment model
  - p_wind_cap_star[w]    : candidate wind capacities from investment model
  - P_BAT_cap_star[b]     : battery power capacities from investment model
  - E_BAT_cap_star[b]     : battery energy capacities from investment model
  - r_cost_up[g]          : upward reserve cost for each generator
  - r_cost_dn[g]          : downward reserve cost for each generator
  - pg_DA[(g,t)]          : DA scheduled generation for each (g,t)
  - loadyearly_vec        : the loadyearly array

Uses global data:
  ref, Gens, CandGens, ExistGens, Lines, CandLines, ExistLines,
  B_line, BATS, RENW, bus_of_wind, bus_of_bat,
  realload, net_load_fixed, pwind_real,
  battery_data, cand_wind_data,
  c_shed_prime, c_cur_prime, eta
Returns:
  - obj_RT::Float64
  - model_RT::Model
"""
wind_multiplier = 0.80 


function solve_RT_for_day(d0::Int;
    T::Int = 24,
    y::Int = 1,
    p_new_star::Dict{Int,Float64},
    f_cand_star::Dict{Int,Float64},    
    p_wind_cap_star::Dict{String,Float64},
    P_BAT_cap_star::Dict{String,Float64},
    E_BAT_cap_star::Dict{String,Float64},
    r_cost_up::Dict{Int,Float64},
    r_cost_dn::Dict{Int,Float64},
    pg_DA::Dict{Tuple{Int,Int},Float64},
    loadyearly_vec::Vector{Float64} = loadyearly
)

    ly = loadyearly_vec[y]

    # ------------------ Build model ------------------
    model_RT = Model(Gurobi.Optimizer)

    # ================== VARIABLES ==================
    @variable(model_RT, va_prime_blnc[i in keys(ref[:bus]), t in 1:T])

    @variable(model_RT, r_up[g in Gens, t in 1:T] >= 0)
    @variable(model_RT, r_down[g in Gens, t in 1:T] >= 0)

    @variable(model_RT, pg_RT[g in Gens, t in 1:T])
    for g in Gens, t in 1:T
        fix(pg_RT[g, t], pg_DA[(g, t)]; force = true)
    end
    @variable(model_RT,
              p_shed_prime_blnc[i in keys(ref[:bus]), t in 1:T] >= 0)
    @variable(model_RT,
              p_cur_prime_blnc[i in keys(ref[:bus]), t in 1:T]  >= 0)

    @variable(model_RT, e_rt[b in BATS, t in 1:T])
    @variable(model_RT, P_BAT_rt[b in BATS, t in 1:T])

    @variable(model_RT, P_BAT_cap_RT[b in BATS] >= 0)
    @variable(model_RT, E_BAT_cap_RT[b in BATS] >= 0)
    for b in BATS
        fix(P_BAT_cap_RT[b], P_BAT_cap_star[b]; force = true)
        fix(E_BAT_cap_RT[b], E_BAT_cap_star[b]; force = true)
    end

    @variable(model_RT, p_new_RT[g in CandGens] >= 0)
    @variable(model_RT, f_cand_RT[l in CandLines])

    @variable(model_RT, p_wind_cap_RT[w in RENW] >= 0)
    for g in CandGens
        fix(p_new_RT[g], p_new_star[g]; force = true)
    end
    for l in CandLines
        fix(f_cand_RT[l], f_cand_star[l]; force = true)
    end
    for w in RENW
        fix(p_wind_cap_RT[w], p_wind_cap_star[w]; force = true)
    end

    @variable(model_RT,
        -ref[:branch][l]["rate_a"] <=
        p_prime_blnc[(l,i,j) in ref[:arcs_from], t in 1:T] <=
        ref[:branch][l]["rate_a"]
    )

     @variable(model_RT, windp_rt[i in RENW, t in 1:T]  >= 0)


    
    p_expr_prime_blnc =
        Dict(((l, i, j), t) => 1.0 * p_prime_blnc[(l, i, j), t]
             for (l, i, j) in ref[:arcs_from], t in 1:T)
    p_expr_prime_blnc = merge(p_expr_prime_blnc,
        Dict(((l, j, i), t) => -1.0 * p_prime_blnc[(l, i, j), t]
            for (l, i, j) in ref[:arcs_from], t in 1:T))

    # ================== CONSTRAINTS ==================

    # --- DC power flow ---
    for l in ExistLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_RT,
            p_prime_blnc[f_idx, t] ==
            -B_line[l] * (va_prime_blnc[f_bus, t] - va_prime_blnc[t_bus, t])
        )
    end
    for l in CandLines, t in 1:T
        br = ref[:branch][l]
        f_bus = br["f_bus"]
        t_bus = br["t_bus"]
        f_idx = (l, f_bus, t_bus)
        @constraint(model_RT,
            p_prime_blnc[f_idx, t] ==
            -B_line[l] * (va_prime_blnc[f_bus, t] - va_prime_blnc[t_bus, t])
        )
    end

    # --- Angle reference ---
    for (i, _bus) in ref[:ref_buses], t in 1:T
        @constraint(model_RT, va_prime_blnc[i, t] == 0)
    end
    

    for l in CandLines, t in 1:T
        br = ref[:branch][l]; f = br["f_bus"]; to = br["t_bus"]; f_idx = (l, f, to)
        @constraint(model_RT, p_prime_blnc[f_idx,t] <=  f_cand_RT[l])
        @constraint(model_RT, -f_cand_RT[l] <=  p_prime_blnc[f_idx,t])
    end

    
    for g in CandGens, t in 1:T
        @constraint(model_RT, r_up[g, t]   <= p_new_star[g])
        @constraint(model_RT, r_down[g, t] <= p_new_star[g])
    end
    for g in ExistGens, t in 1:T
        @constraint(model_RT, r_up[g, t]   <= ref[:gen][g]["pmax"])
        @constraint(model_RT, r_down[g, t] <= ref[:gen][g]["pmax"])
    end

   

    for g in CandGens, t in 1:T
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   <= p_new_star[g])
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   >= 0)
    end
    for g in ExistGens, t in 1:T
        @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t] <= ref[:gen][g]["pmax"])
         @constraint(model_RT, pg_RT[g, t] + r_up[g, t] - r_down[g, t]   >= 0)
    end
 

    #ramp
    for g in CandGens, t in 2:T
        @constraint(model_RT, pg_RT[g,t] + r_up[g,t] - r_down[g,t] - (pg_RT[g, t-1] + r_up[g,t-1] - r_down[g,t-1]) <= 0.8*  p_new_star[g])
        @constraint(model_RT, pg_RT[g,t-1] + r_up[g,t-1] - r_down[g,t-1] - (pg_RT[g,t] + r_up[g,t] - r_down[g,t]) <= 0.8* p_new_star[g])
    end

    for g in ExistGens, t in 2:T
        @constraint(model_RT, pg_RT[g,t] + r_up[g,t] - r_down[g,t] - (pg_RT[g, t-1] + r_up[g,t-1] - r_down[g,t-1]) <= 0.8 * ref[:gen][g]["pmax"])
        @constraint(model_RT, pg_RT[g,t-1] + r_up[g,t-1] - r_down[g,t-1] - (pg_RT[g,t] + r_up[g,t] - r_down[g,t]) <= 0.8 * ref[:gen][g]["pmax"])
    end


    

    # --- Battery RT constraints ---
    @variable(model_RT, P_DIS_rt[b in BATS, t in 1:T] >= 0)
    @variable(model_RT, P_CH_rt[b in BATS, t in 1:T] >= 0)
    for b in BATS, t in 1:T
        @constraint(model_RT, P_BAT_rt[b,t] == P_DIS_rt[b,t] - P_CH_rt[b,t])
    end
    
    for b in BATS, t in 1:T
        @constraint(model_RT, e_rt[b, t] <= E_BAT_cap_star[b])
        @constraint(model_RT, 0 <= e_rt[b, t])

        @constraint(model_RT, P_BAT_rt[b, t] <= P_BAT_cap_star[b])
        @constraint(model_RT, -P_BAT_cap_star[b] <= P_BAT_rt[b, t])
    end

    for b in BATS, t in 2:T
        @constraint(model_RT, e_rt[b, t] == e_rt[b, t-1] - P_BAT_rt[b, t])
    end


    
    @constraint(model_RT, [b in BATS],
        e_rt[b, 1] == 0.5 * E_BAT_cap_star[b]
    )
    @constraint(model_RT, [b in BATS],
        e_rt[b, T] == 0.5 * E_BAT_cap_star[b]
    )


for g in RENW, t in 1:T
    bus = bus_of_wind[g]
    @constraint(model_RT,
        windp_rt[g,t] <= (pwind_real[bus][t][d0][y] * 0.80 / 400) * p_wind_cap_star[g]
    )
    @constraint(model_RT, 0 <= windp_rt[g,t])
end

    


    
    # --- Nodal balance with RT redispatch ---
    for (i, _bus) in ref[:bus], t in 1:T
        @constraint(model_RT,
            sum(p_expr_prime_blnc[a, t] for a in ref[:bus_arcs][i]) ==
            sum(pg_RT[g, t] + r_up[g, t] - r_down[g, t]
                for g in ref[:bus_gens][i])
            + sum(P_BAT_rt[b, t] for b in BATS if bus_of_bat[b] == i)
            - (ly * (loadd[d0][t][i][y]/100) + ly * net_load_fixed[d0][t][i])
            + sum(windp_rt[w,t] for w in RENW if bus_of_wind[w] == i)
            + p_shed_prime_blnc[i, t] 
        )
    end
for (i,_) in ref[:bus], t in 1:T
        @constraint(model_RT, p_shed_prime_blnc[i,t] <= ly * (loadd[d0][t][i][y]/100) + ly * net_load_fixed[d0][t][i])
    end

    for t in 1:T, (i,_) in ref[:bus]
    gens_at_bus = [g for g in RENW if bus_of_wind[g] == i]

    if isempty(gens_at_bus)
        @constraint(model_RT, p_cur_prime_blnc[i,t] == 0)
    else
        @constraint(model_RT,
            p_cur_prime_blnc[i,t] ==
            sum((pwind_real[i][t][d0][y] * 0.80  / 400) * p_wind_cap_star[g] - windp_rt[g,t]
                for g in gens_at_bus)
        )
    end
end


    # ================== OBJECTIVE ==================

    @objective(model_RT, Min,
        sum(r_cost_up[g] * r_up[g, t] - r_cost_dn[g] * r_down[g, t]
            for g in Gens, t in 1:T)
        +
        sum(c_shed_prime * p_shed_prime_blnc[i, t] +
            c_cur_prime  * p_cur_prime_blnc[i, t]
            for i in keys(ref[:bus]), t in 1:T)
        +
        sum(battery_data[b].c_opr_b * (P_DIS_rt[b, t] + P_CH_rt[b,t])
            for b in BATS, t in 1:T)
    )

    # ------------------ Solve ------------------
    optimize!(model_RT)

rt_up_cost = value(sum(r_cost_up[g] * r_up[g,t] for g in Gens, t in 1:T))
rt_dn_credit = value(sum(r_cost_dn[g] * r_down[g,t] for g in Gens, t in 1:T))
rt_shed_cost = value(sum(c_shed_prime * p_shed_prime_blnc[i,t] for i in keys(ref[:bus]), t in 1:T))
rt_cur_cost = value(sum(c_cur_prime * p_cur_prime_blnc[i,t] for i in keys(ref[:bus]), t in 1:T))
rt_bat_cost = value(sum(battery_data[b].c_opr_b * (P_DIS_rt[b,t] + P_CH_rt[b,t]) for b in BATS, t in 1:T))

println("RT day $d0 up    = ", rt_up_cost)
println("RT day $d0 down  = ", rt_dn_credit)
println("RT day $d0 shed  = ", rt_shed_cost)
println("RT day $d0 cur   = ", rt_cur_cost)
println("RT day $d0 bat   = ", rt_bat_cost)
println("RT day $d0 sum   = ", rt_up_cost - rt_dn_credit + rt_shed_cost + rt_cur_cost + rt_bat_cost )



wind_total = sum(value(windp_rt[g, t]) for g in RENW, t in 1:T)

fuel_total = sum(
    value(pg_RT[g, t] + r_up[g, t] - r_down[g, t])
    for g in Gens, t in 1:T
)

ratio = fuel_total > 0 ? wind_total / fuel_total  : NaN

obj_RT = objective_value(model_RT)

println("RT day $d0 ratio    = ", ratio)    
println("RT day $d0 fuel_total   = ", fuel_total)  
println("RT day $d0 wind_total   = ", wind_total)  
    
return obj_RT, wind_total, fuel_total, ratio
end





# OOS test

In [ ]:
N_total_days = 731

# Days we already used
used_days = collect(1:2:TotalDays)   # [1,3,5,...,99]

# All possible days
all_days = collect(320:2:N_total_days)


sampled_days = all_days


DD = length(sampled_days)


println("Sampled days = ", sampled_days)

DA_obj = zeros(Float64, DD)
pg_all_days = Vector{Dict{Tuple{Int,Int},Float64}}(undef, DD)
#####################
# DAY-AHEAD PROBLEM 


for (i, d0) in enumerate(sampled_days)
    println("=== DA for day $d0 (sample index $i) ===")
    obj_DA, pg_DA, model_DA = solve_DA_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
    )
    DA_obj[i] = obj_DA
    pg_all_days[i] = pg_DA
end

total_DA = sum(DA_obj)
println("Total DA cost over sampled days = ", total_DA)

#####################
# REAL-TIME PROBLEM 


RT_obj = zeros(Float64, DD)
wind_totals = zeros(Float64, DD)
fuel_totals = zeros(Float64, DD)
ratios = zeros(Float64, DD)

for (i, d0) in enumerate(sampled_days)
    println("=== RT for day $d0 (sample index $i) ===")

    pg_DA = pg_all_days[i]

    obj_RT, wind_total, fuel_total, ratio = solve_RT_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
        r_cost_up       = r_cost_up,
        r_cost_dn       = r_cost_dn,
        pg_DA           = pg_DA,
    )

    RT_obj[i] = obj_RT
    wind_totals[i] = wind_total
    fuel_totals[i] = fuel_total
    ratios[i] = ratio
end

total_RT = sum(RT_obj)
total_wind_total = sum(wind_totals)
total_fuel_totals = sum(fuel_totals)
ratio_total = total_wind_total / total_fuel_totals

println("Total DA cost over sampled days = ", total_DA)
println("Total RT cost over sampled days = ", total_RT)
println("Total system cost (DA + RT) over sampled days = ", total_DA + total_RT)
println("total_wind_total over sampled days = ", total_wind_total)
println("total_fuel_totals over sampled days = ", total_fuel_totals)
println("ratio_total over sampled days = ", ratio_total)


# --- Cost components (y = 1) ---

gen_cost = sum(p_new_star[i] * c_g_var_id[i] for i in CandGens)

line_cost = sum(f_cand_star[l] * c_l_var_id[l] for l in CandLines)

wind_cost = sum(p_wind_cap_star[g] * cand_wind_data[g].c_var for g in RENW)

bat_energy_cost = sum(E_BAT_cap_star[g] * battery_data[g].c_var_E for g in BATS)

# --- Total  ---
total_cost = gen_cost + line_cost + wind_cost + bat_energy_cost

println("\n=== Investment Cost Breakdown PK (y = 1) ===")
println("Candidate Generators: ", gen_cost)
println("Candidate Lines:      ", line_cost)
println("Wind:                 ", wind_cost)
println("Battery Energy:       ", bat_energy_cost)
println("----------------------------------------")
println("Total:                ", total_cost)

In [ ]:
# === Sets ===
CandGens  = [34, 35, 36, 37, 38, 39, 40]
CandLines = [39, 40]

RENW = [
    "Wind Gen 1",
    "Wind Gen 2",
    "Wind Gen 3",
    "Wind Gen 4",
    "Wind Gen 5",
    "Wind Gen 6",
]

BATS = ["Battery 7", "Battery 16", "Battery 20", "Battery 24"]

# === NEW OPTIMAL VALUES ===

p_new_star = Dict(
    34 => 4.118914163098104,
    35 => 3.499007407934777,
    36 => 4.745107707294619,
    37 => 2.797981903829773,
    38 => 3.0997391096859754,
    39 => 3.148547035342584,
    40 => 2.198141270185045,
)

f_cand_star = Dict(
    39 => 1.799065790901486,
    40 => 1.173610882981401,
)

p_wind_cap_star = Dict(
    "Wind Gen 1" => 15.504009567986653,
    "Wind Gen 2" => 12.223483425587162,
    "Wind Gen 3" => 17.34287593660864,
    "Wind Gen 4" => 12.363535069470126,
    "Wind Gen 5" => 17.174073637879186,
    "Wind Gen 6" => 11.293744426359334,
)

P_BAT_cap_star = Dict(
    "Battery 7"  => 0.39997639088155423,
    "Battery 16" => 1.5998494230962634,
    "Battery 20" => 0.95,
    "Battery 24" => 0.499933831594649,
)

E_BAT_cap_star = Dict(
    "Battery 7"  => 1.599905563526217,
    "Battery 16" => 6.399397692385054,
    "Battery 20" => 3.8,
    "Battery 24" => 1.999735326378596,
)

# === Sanity checks ===
@assert all(haskey(p_new_star, g) for g in CandGens)
@assert all(haskey(f_cand_star, l) for l in CandLines)
@assert all(haskey(p_wind_cap_star, w) for w in RENW)
@assert all(haskey(P_BAT_cap_star, b) for b in BATS)
@assert all(haskey(E_BAT_cap_star, b) for b in BATS)

println("All sanity checks passed.")

# === Battery duration check ===
println("\nBattery duration check (E/P):")
for b in BATS
    if P_BAT_cap_star[b] > 0
        println("$b → ", round(E_BAT_cap_star[b] / P_BAT_cap_star[b], digits=4), " hours")
    else
        println("$b → no battery installed")
    end
end

# === Summary ===
println("\n=== SUMMARY ===")
println("Total new generation: ", sum(values(p_new_star)))
println("Total line capacity:  ", sum(values(f_cand_star)))
println("Total wind capacity:  ", sum(values(p_wind_cap_star)))
println("Total battery power:  ", sum(values(P_BAT_cap_star)))
println("Total battery energy: ", sum(values(E_BAT_cap_star)))

# === Final output ===
println("\n=== FINAL SOLUTION ===")
println("p_new*:       ", p_new_star)
println("f_cand*:      ", f_cand_star)
println("p_wind_cap*:  ", p_wind_cap_star)
println("P_BAT_cap*:   ", P_BAT_cap_star)
println("E_BAT_cap*:   ", E_BAT_cap_star)

In [ ]:
println("Sampled days = ", sampled_days)

DA_obj = zeros(Float64, DD)
pg_all_days = Vector{Dict{Tuple{Int,Int},Float64}}(undef, DD)
#####################
# DAY-AHEAD PROBLEM 

for (i, d0) in enumerate(sampled_days)
    println("=== DA for day $d0 (sample index $i) ===")
    obj_DA, pg_DA, model_DA = solve_DA_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
    )
    DA_obj[i] = obj_DA
    pg_all_days[i] = pg_DA
end

total_DA = sum(DA_obj)
println("Total DA cost over sampled days = ", total_DA)

#####################
# REAL-TIME PROBLEM 

RT_obj = zeros(Float64, DD)
wind_totals = zeros(Float64, DD)
fuel_totals = zeros(Float64, DD)
ratios = zeros(Float64, DD)

for (i, d0) in enumerate(sampled_days)
    println("=== RT for day $d0 (sample index $i) ===")

    pg_DA = pg_all_days[i]

    obj_RT, wind_total, fuel_total, ratio = solve_RT_for_day(
        d0;
        p_new_star      = p_new_star,
        f_cand_star     = f_cand_star,
        p_wind_cap_star = p_wind_cap_star,
        P_BAT_cap_star  = P_BAT_cap_star,
        E_BAT_cap_star  = E_BAT_cap_star,
        r_cost_up       = r_cost_up,
        r_cost_dn       = r_cost_dn,
        pg_DA           = pg_DA,
    )

    RT_obj[i] = obj_RT
    wind_totals[i] = wind_total
    fuel_totals[i] = fuel_total
    ratios[i] = ratio
end

total_RT = sum(RT_obj)
total_wind_total = sum(wind_totals)
total_fuel_totals = sum(fuel_totals)
ratio_total = total_wind_total / total_fuel_totals





println("Total DA cost over sampled days = ", total_DA)
println("Total RT cost over sampled days = ", total_RT)
println("Total system cost (DA + RT) over sampled days = ", total_DA + total_RT)
println("total_wind_total over sampled days = ", total_wind_total)
println("total_fuel_totals over sampled days = ", total_fuel_totals)
println("ratio_total over sampled days = ", ratio_total)




# --- Cost components trilevel (y = 1) ---

gen_cost = sum(p_new_star[i] * c_g_var_id[i] for i in CandGens)

line_cost = sum(f_cand_star[l] * c_l_var_id[l] for l in CandLines)

wind_cost = sum(p_wind_cap_star[g] * cand_wind_data[g].c_var for g in RENW)

bat_energy_cost = sum(E_BAT_cap_star[g] * battery_data[g].c_var_E for g in BATS)

# --- Total (consistent with your model) ---
total_cost = gen_cost + line_cost + wind_cost + bat_energy_cost

println("\n=== Investment Cost Breakdown trilevel (y = 1) ===")
println("Candidate Generators: ", gen_cost)
println("Candidate Lines:      ", line_cost)
println("Wind:                 ", wind_cost)
println("Battery Energy:       ", bat_energy_cost)
println("----------------------------------------")
println("Total:                ", total_cost)